# Pharmaceutical Document Q&A

A retrieval-augmented generation pipeline for pharmaceutical "blob" PDFs: single files that
bundle a Cover Letter, Certificates of Quality, Packaging Specification, BSE/TSE Declaration,
Material Description, Supplier Qualification Record and Chain of Custody document together,
with no index and no separators between them.

The system splits the bundle back into its parts, classifies each one, indexes them, and
answers plain-English questions with citations back to the exact document and page.

```
Document Input → OCR → Chunking → Embeddings → Vector Storage → Retrieval → LLM Response
```

Everything runs locally on the Colab runtime. No API key, no external account, no document
text leaving the machine.

| | |
|---|---|
| **LLM** | Qwen2.5-3B-Instruct (local, Hugging Face Transformers) |
| **Embeddings** | all-MiniLM-L6-v2, 384-dimensional |
| **Vector store** | FAISS, cosine similarity |
| **OCR** | Tesseract 5, for scanned pages |
| **UI** | Gradio |

### How to run

1. Set the runtime to a GPU: **Runtime → Change runtime type → T4 GPU**. It will run on CPU,
   just slowly.
2. Upload `pharma-blob-sample.pdf` (or your own bundled PDF) — you can do this through the
   UI once it launches.
3. **Runtime → Run all.** Step 2 downloads the models, which is the slow part (~3 minutes).
4. Open the Gradio link from the final cell, upload the PDF, click **Process Document**,
   and ask questions.

### Try these

- *"What is the lot number and expiration date for the AKTA ready Low Flow Kit?"* — a
  straightforward lookup, but there are **two** certificates in the file, so the retriever
  has to pick the right one.
- *"What is the lot number and expiration date for part number 29477427?"* — that part is
  a real product referenced across five of the seven documents, but its Certificate of
  Quality is missing from the bundle. The system should say so rather than borrowing a lot
  number from a neighbouring certificate.

> The evaluation that produced the metrics in the README lives in a separate notebook,
> `02_evaluation.ipynb`, so this one stays focused on the pipeline itself.

## Setup

In [ ]:
# ============================================
# STEP 1: Install Dependencies
# ============================================
# Everything the pipeline needs: Gradio for the UI, PyMuPDF for reading PDFs,
# Tesseract for OCR on scanned pages, FAISS for vector search, Sentence
# Transformers for embeddings, and Transformers for the local LLM.
#
# Takes 2-3 minutes on a fresh runtime. Anything Colab already has is skipped.
# ============================================

# System dependency for OCR (scanned PDFs)
!apt-get -qq update && apt-get -qq install -y tesseract-ocr

!pip install -q gradio
!pip install -q pypdf PyPDF2 pymupdf
!pip install -q pytesseract pillow
!pip install -q sentence-transformers faiss-cpu
!pip install -q numpy pandas
!pip install -q "transformers>=4.46" accelerate

# Optional: LlamaIndex powers the alternative sentence-aware chunker in Step 6.
# Comment this out if you only need the default chunker.
!pip install -q llama-index llama-index-embeddings-huggingface

print("Setup complete. Restart the runtime only if Colab explicitly asks you to.")

## Imports, Models and Configuration

In [ ]:
# ============================================
# STEP 2: Imports, Models and Configuration
# ============================================
# Loads the two models the pipeline runs on, both locally, with no API key
# and no external account:
#   - Qwen2.5-3B-Instruct  : classifies documents, routes queries, writes answers
#   - all-MiniLM-L6-v2     : turns text into vectors for search
#
# Also sets up the stopwatch (TIMING) that records how long each stage of the
# pipeline takes. Those numbers feed the performance table in the evaluation.
#
# You'll see: download bars for both models, then a confirmation line.
# ============================================

import gradio as gr
import fitz  # PyMuPDF
from PyPDF2 import PdfReader
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer
import faiss
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from typing import List, Dict, Tuple, Optional, Sequence
from dataclasses import dataclass, field
import json
import re
import os
import io
import time
from pathlib import Path
from contextlib import contextmanager
from collections import defaultdict
from datetime import datetime
import hashlib

from PIL import Image, ImageFilter
import pytesseract

# LlamaIndex imports for enhanced document processing
from llama_index.core import Document, VectorStoreIndex, StorageContext
from llama_index.core.schema import TextNode
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core.node_parser import SentenceSplitter
from llama_index.core.vector_stores import MetadataFilters, MetadataFilter, FilterOperator

# ------------------------------------------------------------------
# Open-source LLM (replaces Gemini)
# ------------------------------------------------------------------
# Swap this for a bigger model (e.g. "Qwen/Qwen2.5-7B-Instruct") if you
# have an A100 runtime, or a smaller one (e.g. "Qwen/Qwen2.5-1.5B-Instruct")
# for a faster, lighter demo. No Hugging Face login is required for any
# of the Qwen2.5-Instruct models.
LLM_MODEL_NAME = "Qwen/Qwen2.5-3B-Instruct"
EMBED_MODEL_NAME = "all-MiniLM-L6-v2"

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Loading {LLM_MODEL_NAME} on {device}...")

llm_tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL_NAME)
llm_model = AutoModelForCausalLM.from_pretrained(
    LLM_MODEL_NAME,
    torch_dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto" if device == "cuda" else None,
)
if device == "cpu":
    llm_model.to(device)


# ------------------------------------------------------------------
# Stage-level timing instrumentation
# ------------------------------------------------------------------
# TIMING accumulates total seconds spent per pipeline stage; TIMING_COUNTS
# tracks how many calls contributed to each. Call reset_timing() before
# processing each document if you want per-file (rather than cumulative)
# breakdowns. The stages recorded are:
#   classification, boundary_detection, ocr, embedding, indexing,
#   routing, retrieval, answer_generation
TIMING = defaultdict(float)
TIMING_COUNTS = defaultdict(int)


def reset_timing():
    TIMING.clear()
    TIMING_COUNTS.clear()


@contextmanager
def timed(stage: str):
    """Record wall time for a pipeline stage. Used everywhere instead of
    hand-rolled t0/elapsed pairs so no stage gets missed."""
    t0 = time.time()
    try:
        yield
    finally:
        TIMING[stage] += time.time() - t0
        TIMING_COUNTS[stage] += 1


def timing_table(stage_timing: dict, stage_counts: dict, total_wall: float = None) -> pd.DataFrame:
    """Turn the raw timing dictionaries into a readable DataFrame."""
    rows = []
    for stage, secs in sorted(stage_timing.items(), key=lambda x: -x[1]):
        n = stage_counts.get(stage, 0)
        rows.append({
            "Stage": stage,
            "Total (s)": round(secs, 2),
            "Calls": n,
            "Avg (s/call)": round(secs / max(n, 1), 3),
        })
    if total_wall is not None:
        accounted = sum(stage_timing.values())
        rows.append({
            "Stage": "(unaccounted / overhead)",
            "Total (s)": round(max(total_wall - accounted, 0), 2),
            "Calls": None,
            "Avg (s/call)": None,
        })
    return pd.DataFrame(rows)


def generate_text(prompt: str, max_new_tokens: int = 300, stage: str = "other") -> str:
    """
    Single entry point for every LLM call in the pipeline (document
    classification, boundary detection, query routing, and answer
    generation). Keeping all model calls behind this one function means
    swapping the underlying open-source model only requires changing it
    here and in LLM_MODEL_NAME above. The `stage` label lets us track how
    much time each part of the pipeline spends in the LLM (see TIMING).
    """
    with timed(stage):
        messages = [{"role": "user", "content": prompt}]
        chat_prompt = llm_tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
        inputs = llm_tokenizer(chat_prompt, return_tensors="pt").to(llm_model.device)

        with torch.no_grad():
            output_ids = llm_model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,  # greedy decoding: deterministic, good for extraction/classification
                pad_token_id=llm_tokenizer.eos_token_id,
            )

        generated_ids = output_ids[0][inputs["input_ids"].shape[1]:]
        result = llm_tokenizer.decode(generated_ids, skip_special_tokens=True).strip()

    return result


# ------------------------------------------------------------------
# Embedding model
# ------------------------------------------------------------------
# all-MiniLM-L6-v2 is trained with a cosine objective, so embeddings are
# L2-normalised before indexing and FAISS inner-product search is used.
# This makes the similarity scores true cosine similarities in [0, 1],
# which is what the UI reports as "relevance".
embed_model = SentenceTransformer(EMBED_MODEL_NAME)
llama_embed_model = HuggingFaceEmbedding(model_name=f"sentence-transformers/{EMBED_MODEL_NAME}")


def embed_texts(texts, batch_size: int = 32) -> np.ndarray:
    """Encode and L2-normalise, returning float32 for FAISS."""
    vecs = embed_model.encode(
        list(texts), batch_size=batch_size, show_progress_bar=False,
        normalize_embeddings=True,
    )
    return np.asarray(vecs, dtype="float32")


print(f"Imports and configuration complete. LLM: {LLM_MODEL_NAME} | Embeddings: {EMBED_MODEL_NAME} | Device: {device}")

## Data Structures

In [ ]:
# ============================================
# STEP 3: Data Structures
# ============================================
# Three simple containers that carry information through the pipeline:
#   PageInfo        - one PDF page
#   LogicalDocument - one sub-document found inside the blob PDF
#   ChunkMetadata   - one searchable passage, plus where it came from
#
# The "where it came from" part is what lets every answer cite a document
# type, a page range and a filename.
#
# You'll see: no output. These are just definitions.
# ============================================

@dataclass
class PageInfo:
    """Stores information about a single page"""
    page_num: int
    text: str
    doc_type: Optional[str] = None
    page_in_doc: int = 0
    text_source: str = "digital"   # "digital" | "ocr" | "empty"
    char_count: int = 0


@dataclass
class LogicalDocument:
    """Represents a logical document within a PDF"""
    doc_id: str
    doc_type: str
    page_start: int
    page_end: int
    text: str
    source_file: Optional[str] = None
    page_texts: List[Tuple[int, str]] = field(default_factory=list)
    text_source: str = "digital"
    chunks: List["ChunkMetadata"] = field(default_factory=list)


@dataclass
class ChunkMetadata:
    """Rich metadata for each chunk"""
    chunk_id: str
    doc_id: str
    doc_type: str
    chunk_index: int
    page_start: int
    page_end: int
    text: str
    source_file: Optional[str] = None
    text_source: str = "digital"
    embedding: Optional[np.ndarray] = None

    @property
    def pages_1indexed(self) -> str:
        """Page range as humans count pages. Used in every citation and in
        the UI so the two never disagree."""
        return f"{self.page_start + 1}-{self.page_end + 1}"

## Document Intelligence

The hard part of a bundled PDF is that nothing marks where one document ends and the next
begins. These two functions work it out from the page content itself.

In [ ]:
# ============================================
# STEP 4: Document Classification and Boundary Detection
# ============================================
# A blob PDF is many documents stapled into one file. Before we can answer
# anything accurately we have to work out where each sub-document starts and
# what kind of document it is.
#
#   classify_document_type()   - which of the 7 types is this page?
#   detect_document_boundary() - are these two pages the same document?
#
# Getting this right matters: a question about lot numbers should pull from
# the Certificate of Quality, not the Cover Letter.
#
# You'll see: no output. Both are called during PDF processing in Step 5.
# ============================================

VALID_DOC_TYPES = [
    "Cover Letter", "Certificate Of Quality", "Packaging Specification",
    "Bse/Tse Declaration", "Material Description", "Supplier Qualification",
    "Chain Of Custody", "Other"
]

# Aliases the model tends to produce for each canonical label. Checked
# before the substring match so that e.g. "Chain of Custody Record" or
# "TSE Statement" resolve correctly instead of becoming a brand-new label.
DOC_TYPE_ALIASES = {
    "Cover Letter": ["cover letter", "letter", "to whom it may concern"],
    "Certificate Of Quality": ["certificate of quality", "certificate of analysis",
                               "coq", "quality certificate", "certificate"],
    "Packaging Specification": ["packaging specification", "packaging spec",
                                "package specification", "packaging"],
    "Bse/Tse Declaration": ["bse/tse", "bse tse", "tse declaration", "bse declaration",
                            "transmissible spongiform", "animal origin"],
    "Material Description": ["material description", "materials of construction",
                             "material data", "material sheet"],
    "Supplier Qualification": ["supplier qualification", "supplier audit",
                               "supplier record", "vendor qualification"],
    "Chain Of Custody": ["chain of custody", "traceability", "custody"],
    "Other": ["other", "unknown", "n/a"],
}


def clean_doc_type(response: str) -> str:
    """
    Map a free-text LLM response onto one of VALID_DOC_TYPES.

    Anything that cannot be matched falls back to "Other" rather than
    being passed through verbatim. Passing it through would create
    one-off document types that fragment the per-type indices and make
    the routing metrics meaningless.
    """
    cleaned = (response or "").strip().lower()
    cleaned = cleaned.replace('"', "").replace("`", "").replace("*", "").replace(".", "").strip()
    if not cleaned:
        return "Other"

    # Exact label match first.
    for label in VALID_DOC_TYPES:
        if cleaned == label.lower():
            return label

    # Then alias / substring match, longest alias first so that
    # "certificate of quality" wins over the bare alias "certificate".
    scored = []
    for label, aliases in DOC_TYPE_ALIASES.items():
        for alias in aliases:
            if alias in cleaned:
                scored.append((len(alias), label))
    if scored:
        return max(scored)[1]

    return "Other"


def classify_document_type(text: str, max_length: int = 1500) -> str:
    """
    Classify the document type based on its content.
    Uses the local open-source LLM to identify the document category.
    """
    if not text or not text.strip():
        return "Other"

    text_sample = text[:max_length] if len(text) > max_length else text

    prompt = f"""You are a document classifier. Based on the page
content below, classify it into ONE of these document types:

- Cover Letter: A formal letter (often starting with "To Whom It
  May Concern") discussing product information or storage conditions.
- Certificate of Quality: Contains lot numbers, manufacture dates,
  expiration dates, and test results (autoclave, gamma irradiation).
- Packaging Specification: Describes packaging components, materials,
  part numbers, and configuration change history.
- BSE/TSE Declaration: A declaration about animal-origin materials
  and transmissible spongiform encephalopathy compliance.
- Material Description: Lists materials of construction, sterilization
  compatibility, and physical properties of a product.
- Supplier Qualification: Contains supplier audit history,
  certifications (ISO 9001, ISO 13485), and approved product lists.
- Chain of Custody: Lists manufactured assemblies, traceability
  information, and the manufacturing-to-shipment flow.
- Other: Use ONLY if the content does not match any of the above.

Page content:
{text_sample}

Respond with ONLY the document type name. No explanation."""

    try:
        response_text = generate_text(prompt, max_new_tokens=20, stage="classification")
        return clean_doc_type(response_text)
    except Exception as e:
        print(f"Classification error: {e}")
        return "Other"


def detect_document_boundary(prev_text: str, curr_text: str,
                             current_doc_type: str = None) -> bool:
    """
    Detect if two consecutive pages belong to the same document.
    Returns True if they're from the same document.

    If either page has no usable text (a blank page, or a scanned page
    where OCR returned nothing) we keep the pages together rather than
    forcing a split. A blank page is far more likely to be a continuation
    than the start of a genuinely new document, and splitting there used
    to create empty single-page "documents".
    """
    if not prev_text or not prev_text.strip() or not curr_text or not curr_text.strip():
        return True

    prev_sample = prev_text[-500:] if len(prev_text) > 500 else prev_text
    curr_sample = curr_text[:500] if len(curr_text) > 500 else curr_text

    prompt = f"""Determine if these two pages are from the SAME document.

Current document type: {current_doc_type or 'Unknown'}

A NEW document starts when the page has:
- A different document title or heading (e.g., "Certificate of Quality"
  vs "Packaging Specification" vs "Material Description Sheet")
- A completely different topic or subject matter
- Its own header with a new document number or reference

Pages belong to the SAME document when:
- The second page says "continued" or "page 2 of 2"
- The content directly continues the previous page's discussion
- They share the same document number or title

End of Previous Page:
...{prev_sample}

Start of Current Page:
{curr_sample}...

Answer ONLY 'Yes' if same document or 'No' if different document."""

    try:
        response_text = generate_text(prompt, max_new_tokens=10, stage="boundary_detection")
        return response_text.strip().lower().startswith("yes")
    except Exception as e:
        print(f"Boundary detection error: {e}")
        # Default to keeping pages together if uncertain
        return True

## Document Input and OCR

Pages with a readable text layer are read directly. Pages without one — scans — are
rendered to images and passed through Tesseract.

In [ ]:
# ============================================
# STEP 5: Document Input and OCR
# ============================================
# Opens the PDF and reads each page. If a page has little or no readable
# text -- which is what a scanned page looks like -- it is rendered to an
# image and passed to Tesseract OCR instead.
#
# Every page records whether its text came from the PDF's text layer or from
# OCR. That single flag is what lets the evaluation compare digital and
# scanned performance later on.
#
# The pages are then grouped into logical sub-documents using the
# classification and boundary functions from Step 4.
#
# You'll see: no output. This runs inside process_pdf() in Step 9.
# ============================================

# A page with fewer than this many characters of extractable text is
# treated as scanned and sent to OCR.
OCR_MIN_CHARS = 40
OCR_DPI = 200


def _ocr_page(page) -> str:
    """Rasterise a PDF page and run Tesseract on it. Timed as the 'ocr' stage."""
    with timed("ocr"):
        try:
            pix = page.get_pixmap(dpi=OCR_DPI)
            img = Image.open(io.BytesIO(pix.tobytes("png")))
            return pytesseract.image_to_string(img)
        except Exception as e:
            print(f"    OCR failed: {e}")
            return ""


def _open_pdf(pdf_file):
    """Accept a path, a file-like object, or a Gradio dict payload."""
    if isinstance(pdf_file, dict) and "content" in pdf_file:
        return fitz.open(stream=pdf_file["content"], filetype="pdf")
    if hasattr(pdf_file, "read"):
        return fitz.open(stream=pdf_file.read(), filetype="pdf")
    return fitz.open(pdf_file)


def extract_and_analyze_pdf(pdf_file, filename: str = "document.pdf",
                            verbose: bool = True) -> Tuple[List[PageInfo], List[LogicalDocument]]:
    """
    Extract text from PDF and perform intelligent document analysis.
    Returns both page-level info and logical document groupings.
    Supports digital PDFs and scanned PDFs (via OCR).
    """
    if verbose:
        print("Starting PDF extraction and analysis...")

    doc = _open_pdf(pdf_file)

    pages_info = []
    for i, page in enumerate(doc):
        text = page.get_text()
        source = "digital"

        if len(text.strip()) < OCR_MIN_CHARS:
            if verbose:
                print(f"  Page {i}: text layer has {len(text.strip())} chars, running OCR...")
            ocr_text = _ocr_page(page)
            if len(ocr_text.strip()) > len(text.strip()):
                text, source = ocr_text, "ocr"
                if verbose:
                    print(f"  Page {i}: OCR extracted {len(text.strip())} characters")
            else:
                source = "empty" if not text.strip() else "digital"
                if verbose:
                    print(f"  Page {i}: OCR produced no improvement")

        pages_info.append(PageInfo(
            page_num=i, text=text, text_source=source, char_count=len(text.strip())
        ))

    doc.close()

    if not pages_info:
        raise ValueError("PDF contains no pages")
    if all(p.char_count == 0 for p in pages_info):
        raise ValueError("No text could be extracted from PDF (text layer empty and OCR returned nothing)")

    n_ocr = sum(1 for p in pages_info if p.text_source == "ocr")
    if verbose:
        print(f"Extracted {len(pages_info)} pages ({n_ocr} via OCR, {len(pages_info) - n_ocr} from text layer)")
        print("Analyzing document structure...")

    def _finalise(doc_counter, current_doc_type, current_doc_pages) -> LogicalDocument:
        sources = {p.text_source for p in current_doc_pages}
        return LogicalDocument(
            doc_id=f"doc_{doc_counter}",
            doc_type=current_doc_type,
            page_start=current_doc_pages[0].page_num,
            page_end=current_doc_pages[-1].page_num,
            text="\n\n".join(p.text for p in current_doc_pages),
            source_file=filename,
            page_texts=[(p.page_num, p.text) for p in current_doc_pages],
            text_source=("ocr" if "ocr" in sources else "digital"),
        )

    logical_docs = []
    current_doc_type = None
    current_doc_pages = []
    doc_counter = 0

    for i, page_info in enumerate(pages_info):
        if i == 0:
            current_doc_type = classify_document_type(page_info.text)
            page_info.doc_type = current_doc_type
            page_info.page_in_doc = 0
            current_doc_pages = [page_info]
            if verbose:
                print(f"  Page {i}: New document detected - {current_doc_type}")
            continue

        is_same = detect_document_boundary(pages_info[i - 1].text, page_info.text, current_doc_type)

        if is_same:
            page_info.doc_type = current_doc_type
            page_info.page_in_doc = len(current_doc_pages)
            current_doc_pages.append(page_info)
        else:
            logical_docs.append(_finalise(doc_counter, current_doc_type, current_doc_pages))
            doc_counter += 1

            current_doc_type = classify_document_type(page_info.text)
            page_info.doc_type = current_doc_type
            page_info.page_in_doc = 0
            current_doc_pages = [page_info]
            if verbose:
                print(f"  Page {i}: New document detected - {current_doc_type}")

    if current_doc_pages:
        logical_docs.append(_finalise(doc_counter, current_doc_type, current_doc_pages))

    if verbose:
        print(f"Identified {len(logical_docs)} logical documents")
        for ld in logical_docs:
            print(f"   - {ld.doc_type}: Pages {ld.page_start + 1}-{ld.page_end + 1} "
                  f"({len(ld.text.split())} words, {ld.text_source})")

    return pages_info, logical_docs

## Chunking

In [ ]:
# ============================================
# STEP 6: Chunking
# ============================================
# Splits each sub-document into overlapping passages of about 150 words.
# Search works far better on short focused passages than on whole documents,
# and the 40-word overlap stops a fact being cut in half at a chunk boundary.
#
# Why 150: the sub-documents here are short (92-258 words), so a larger chunk
# size collapses most of them into a single chunk and leaves a corpus of only
# ~9 passages -- too few for retrieval scores to mean anything. 150 was checked
# against the answer key to confirm co-dependent facts (a lot number and its
# expiration date, for instance) still land in the same chunk.
#
# Each chunk keeps its document type, page range and source filename, so a
# retrieved passage can always be traced back to a specific page.
#
# You'll see: no output here. During processing you'll see lines like
#   Certificate Of Quality: Created 3 chunks (pages 2-4)
# ============================================

CHUNK_SIZE_WORDS = 150
CHUNK_OVERLAP_WORDS = 40


def _build_word_page_map(logical_doc: LogicalDocument) -> List[int]:
    """
    Return a list the same length as the document's word list, where entry
    i is the 0-indexed PDF page that word i came from.

    Falls back to spreading words evenly across the document's page range
    if per-page text was not captured.
    """
    if logical_doc.page_texts:
        page_map = []
        for page_num, page_text in logical_doc.page_texts:
            page_map.extend([page_num] * len(page_text.split()))
        if page_map:
            return page_map

    words = logical_doc.text.split()
    if not words:
        return []
    span = max(logical_doc.page_end - logical_doc.page_start, 0)
    return [
        logical_doc.page_start + min(int(i / len(words) * (span + 1)), span)
        for i in range(len(words))
    ]


def chunk_document_with_metadata(logical_doc: LogicalDocument,
                                 chunk_size: int = CHUNK_SIZE_WORDS,
                                 overlap: int = CHUNK_OVERLAP_WORDS) -> List[ChunkMetadata]:
    """
    Chunk a logical document while preserving rich metadata.
    Uses a sliding window with overlap for better context.
    """
    words = logical_doc.text.split()
    page_map = _build_word_page_map(logical_doc)

    def pages_for(start_idx: int, end_idx: int) -> Tuple[int, int]:
        window = page_map[start_idx:end_idx]
        if not window:
            return logical_doc.page_start, logical_doc.page_end
        return min(window), max(window)

    def make_chunk(idx: int, text: str, p_start: int, p_end: int) -> ChunkMetadata:
        return ChunkMetadata(
            chunk_id=f"{logical_doc.doc_id}_chunk_{idx}",
            doc_id=logical_doc.doc_id,
            doc_type=logical_doc.doc_type,
            chunk_index=idx,
            page_start=p_start,
            page_end=p_end,
            text=text,
            source_file=logical_doc.source_file,
            text_source=logical_doc.text_source,
        )

    if len(words) <= chunk_size:
        p_start, p_end = pages_for(0, len(words))
        return [make_chunk(0, logical_doc.text, p_start, p_end)]

    stride = max(chunk_size - overlap, 1)
    chunks_metadata = []
    for i, start_idx in enumerate(range(0, len(words), stride)):
        end_idx = min(start_idx + chunk_size, len(words))
        p_start, p_end = pages_for(start_idx, end_idx)
        chunks_metadata.append(make_chunk(i, " ".join(words[start_idx:end_idx]), p_start, p_end))
        if end_idx >= len(words):
            break

    return chunks_metadata


def chunk_with_llama_index(logical_doc: LogicalDocument,
                           chunk_size: int = CHUNK_SIZE_WORDS,
                           chunk_overlap: int = CHUNK_OVERLAP_WORDS) -> List[ChunkMetadata]:
    """
    Alternative: use LlamaIndex's SentenceSplitter, which respects sentence
    boundaries instead of cutting at a fixed word count.

    Note the units differ: SentenceSplitter counts tokens, our custom
    chunker counts words. The values are passed through unchanged, so a
    LlamaIndex chunk is roughly 25-30% shorter in words than a custom
    chunk of the same nominal size.
    """
    doc = Document(
        text=logical_doc.text,
        metadata={
            "doc_id": logical_doc.doc_id,
            "doc_type": logical_doc.doc_type,
            "page_start": logical_doc.page_start,
            "page_end": logical_doc.page_end,
            "source_file": logical_doc.source_file,
            "source": f"{logical_doc.doc_type}_document",
        },
    )

    splitter = SentenceSplitter(
        chunk_size=chunk_size,
        chunk_overlap=chunk_overlap,
        paragraph_separator="\n\n",
        separator=" ",
    )
    nodes = splitter.get_nodes_from_documents([doc])

    # Map each node back to real page numbers using the same word-page map
    # the custom chunker uses, by locating the node's text in the document.
    page_map = _build_word_page_map(logical_doc)
    all_words = logical_doc.text.split()
    cursor = 0

    chunks_metadata = []
    for i, node in enumerate(nodes):
        node_words = node.text.split()
        start_idx = cursor
        end_idx = min(cursor + len(node_words), len(all_words))
        window = page_map[start_idx:end_idx]
        p_start = min(window) if window else logical_doc.page_start
        p_end = max(window) if window else logical_doc.page_end
        cursor = max(end_idx - chunk_overlap, cursor + 1)

        chunks_metadata.append(ChunkMetadata(
            chunk_id=f"{logical_doc.doc_id}_chunk_{i}",
            doc_id=logical_doc.doc_id,
            doc_type=logical_doc.doc_type,
            chunk_index=i,
            page_start=p_start,
            page_end=p_end,
            text=node.text,
            source_file=logical_doc.source_file,
            text_source=logical_doc.text_source,
        ))

    return chunks_metadata


def process_all_documents(logical_docs: List[LogicalDocument],
                          use_llama_index: bool = False,
                          verbose: bool = True) -> List[ChunkMetadata]:
    """
    Process all logical documents into chunks with metadata.
    Can use either the custom chunker or LlamaIndex's SentenceSplitter.
    """
    all_chunks = []

    for logical_doc in logical_docs:
        chunks = (chunk_with_llama_index(logical_doc) if use_llama_index
                  else chunk_document_with_metadata(logical_doc))
        logical_doc.chunks = chunks
        all_chunks.extend(chunks)
        if verbose and chunks:
            print(f"  {logical_doc.doc_type}: Created {len(chunks)} chunks "
                  f"(pages {chunks[0].page_start + 1}-{chunks[-1].page_end + 1})")

    return all_chunks

## Embeddings, Vector Storage and Retrieval

Where a question gets matched to the passages most likely to answer it.

In [ ]:
# ============================================
# STEP 7: Embeddings, Vector Storage and Retrieval
# ============================================
# Turns every chunk into a vector and stores it in FAISS -- one index over
# everything, plus one index per document type.
#
# When a question arrives, the LLM first guesses which document type is
# likely to hold the answer ("query routing"). If it is confident, we search
# only that document type's index; if not, we fall back to searching
# everything. Routing trades recall for precision, so the evaluation section
# measures whether it actually helps.
#
# Similarity is cosine similarity in the range 0-1, which is what the UI
# reports as "relevance".
#
# You'll see: no output here. During a query you'll see a line like
#   Query routed to: Certificate Of Quality (confidence: 0.91)
# ============================================

ROUTING_CONFIDENCE_THRESHOLD = 0.7


def predict_query_document_type(query: str) -> Tuple[str, float]:
    """
    Predict which document type is most likely to contain the answer.
    Returns predicted type and confidence score.
    """
    prompt = f"""Analyze this query and predict which document type
would most likely contain the answer.

Query: "{query}"

Choose the MOST LIKELY type from:
- Cover Letter: Formal letters about product information or storage conditions
- Certificate Of Quality: Lot numbers, manufacture/expiration dates, test results
- Packaging Specification: Packaging components, materials, part numbers
- Bse/Tse Declaration: Animal-origin material declarations, TSE compliance
- Material Description: Materials of construction, sterilization compatibility
- Supplier Qualification: Supplier audits, ISO certifications, approved products
- Chain Of Custody: Manufactured assemblies, traceability, shipment flow
- Other: General or unclear queries

Respond with ONLY a JSON object in this exact format, nothing else:
{{"type": "DocumentType", "confidence": 0.85}}

Confidence should be between 0.0 and 1.0."""

    try:
        response_text = generate_text(prompt, max_new_tokens=60, stage="routing")
        # Small open-source models sometimes wrap JSON in extra text --
        # pull out just the {...} object before parsing.
        match = re.search(r"\{.*?\}", response_text, re.DOTALL)
        if not match:
            raise ValueError(f"No JSON object found in response: {response_text!r}")
        result = json.loads(match.group(0))
        predicted = clean_doc_type(result.get("type", "Other"))
        confidence = float(result.get("confidence", 0.5))
        confidence = min(max(confidence, 0.0), 1.0)
        return predicted, confidence
    except Exception as e:
        print(f"Query routing error: {e}")
        return "Other", 0.0


class IntelligentRetriever:
    """
    Vector storage and retrieval with per-document-type indices,
    metadata filtering and LLM query routing.
    """

    def __init__(self):
        self.index = None                # global FAISS index over all chunks
        self.chunks_metadata = []
        self.doc_type_indices = {}       # doc_type -> {'index', 'mapping'}
        self.last_routing = None         # routing decision from the most recent auto-routed query
        self.last_search_scope = None    # 'routed:<type>' | 'filtered:<type>' | 'global'

    def build_indices(self, chunks_metadata: List[ChunkMetadata], verbose: bool = True):
        """Embed every chunk and build the global + per-type FAISS indices."""
        if verbose:
            print("Building vector indices...")
        self.chunks_metadata = chunks_metadata

        if not chunks_metadata:
            raise ValueError("No chunks to index")

        with timed("embedding"):
            embeddings = embed_texts([c.text for c in chunks_metadata])

        for chunk, vec in zip(chunks_metadata, embeddings):
            chunk.embedding = vec

        with timed("indexing"):
            dim = embeddings.shape[1]
            # Inner product on L2-normalised vectors == cosine similarity.
            self.index = faiss.IndexFlatIP(dim)
            self.index.add(embeddings)

            self.doc_type_indices = {}
            for doc_type in sorted({c.doc_type for c in chunks_metadata}):
                type_indices = [i for i, c in enumerate(chunks_metadata) if c.doc_type == doc_type]
                type_index = faiss.IndexFlatIP(dim)
                type_index.add(embeddings[type_indices])
                self.doc_type_indices[doc_type] = {"index": type_index, "mapping": type_indices}

        if verbose:
            print(f"Indexed {len(chunks_metadata)} chunks across "
                  f"{len(self.doc_type_indices)} document types")

    @staticmethod
    def _search(index, mapping, query_vec, k: int) -> List[Tuple[int, float]]:
        """
        Search one FAISS index and return (corpus_index, cosine_score) pairs.

        k is clamped to the number of vectors actually in the index, and
        any -1 padding FAISS returns is dropped, so a small index returns
        fewer results rather than duplicated or wrong ones.
        """
        k_eff = min(k, index.ntotal)
        if k_eff <= 0:
            return []
        scores, ids = index.search(query_vec, k_eff)
        out = []
        for local_id, score in zip(ids[0], scores[0]):
            if local_id < 0:
                continue
            corpus_id = mapping[local_id] if mapping is not None else int(local_id)
            out.append((int(corpus_id), float(np.clip(score, 0.0, 1.0))))
        return out

    def retrieve(self, query: str, k: int = 4,
                 filter_doc_type: Optional[str] = None,
                 auto_route: bool = True,
                 verbose: bool = True) -> List[Tuple[ChunkMetadata, float]]:
        """
        Retrieve relevant chunks with optional filtering and routing.
        Returns (chunk, cosine_similarity) pairs in rank order.

        Also records the routing decision on self.last_routing so callers
        can inspect the model's own predicted type and confidence,
        separately from the final answer's retrieval-based confidence.
        """
        self.last_routing = None
        self.last_search_scope = None

        if self.index is None:
            return []

        with timed("retrieval"):
            query_vec = embed_texts([query])

        if filter_doc_type and filter_doc_type in self.doc_type_indices:
            data = self.doc_type_indices[filter_doc_type]
            with timed("retrieval"):
                hits = self._search(data["index"], data["mapping"], query_vec, k)
            self.last_search_scope = f"filtered:{filter_doc_type}"

        elif auto_route:
            predicted_type, confidence = predict_query_document_type(query)
            if verbose:
                print(f"Query routed to: {predicted_type} (confidence: {confidence:.2f})")
            self.last_routing = {"predicted_type": predicted_type, "confidence": confidence}

            if confidence > ROUTING_CONFIDENCE_THRESHOLD and predicted_type in self.doc_type_indices:
                data = self.doc_type_indices[predicted_type]
                with timed("retrieval"):
                    hits = self._search(data["index"], data["mapping"], query_vec, k)
                self.last_search_scope = f"routed:{predicted_type}"
            else:
                with timed("retrieval"):
                    hits = self._search(self.index, None, query_vec, k)
                self.last_search_scope = "global (routing confidence below threshold)"

        else:
            with timed("retrieval"):
                hits = self._search(self.index, None, query_vec, k)
            self.last_search_scope = "global"

        return [(self.chunks_metadata[i], score) for i, score in hits]

## LLM Response with Source Attribution

In [ ]:
# ============================================
# STEP 8: LLM Response with Source Attribution
# ============================================
# Sends the retrieved passages to the LLM as context and asks it to answer
# using only that context, citing sources in [From DocType, Pages X-Y] form.
#
# Two rules matter here. The model is told to copy the citation labels
# exactly, so citations can be checked automatically. And it is told to say
# plainly when the context does not contain the answer, rather than guessing
# -- in a regulated setting a confident invented answer is the worst outcome.
#
# You'll see: no output here. Answers appear in the chat panel.
# ============================================

NO_ANSWER_STRING = "The provided documents do not contain enough information to answer this question."


def format_citation_label(chunk_meta: ChunkMetadata, include_file: bool = True) -> str:
    """The single place the [From ...] citation format is defined, so the
    prompt, the source list and the citation grader can never drift apart."""
    label = f"[From {chunk_meta.doc_type}, Pages {chunk_meta.pages_1indexed}"
    if include_file and chunk_meta.source_file:
        label += f", Source: {chunk_meta.source_file}"
    return label + "]"


def generate_answer_with_sources(query: str,
                                 retrieved_chunks: List[Tuple[ChunkMetadata, float]]) -> Dict:
    """
    Generate an answer with detailed source attribution.
    """
    if not retrieved_chunks:
        return {
            "answer": NO_ANSWER_STRING,
            "sources": [],
            "confidence": 0.0,
            "chunks_used": 0,
        }

    context_parts = []
    sources = []

    for chunk_meta, score in retrieved_chunks:
        context_parts.append(format_citation_label(chunk_meta))
        context_parts.append(chunk_meta.text)
        context_parts.append("")

        sources.append({
            "chunk_id": chunk_meta.chunk_id,
            "doc_type": chunk_meta.doc_type,
            "pages": chunk_meta.pages_1indexed,
            "source_file": chunk_meta.source_file,
            "text_source": chunk_meta.text_source,
            "similarity": round(float(score), 4),
            "relevance": f"{score:.1%}",
            "preview": chunk_meta.text[:100] + ("..." if len(chunk_meta.text) > 100 else ""),
        })

    context = "\n".join(context_parts)

    prompt = f"""You are answering questions about a set of documents that
include certificates of quality, packaging specifications, and compliance
declarations. Use the provided context to answer the question accurately.
Be specific and cite which document type and pages support your answer,
using the same [From DocType, Pages X-Y] format shown in the context.

Context:
{context}

Question: {query}

Instructions:
1. Answer based ONLY on the provided context.
2. Cite the document type and page(s) that support each part of your answer,
   copying the [From DocType, Pages X-Y] labels exactly as they appear above.
3. Be concise but complete.
4. If the context does not contain the answer, reply exactly:
   "{NO_ANSWER_STRING}"
   Do not guess and do not use outside knowledge.

Answer:"""

    try:
        answer = generate_text(prompt, max_new_tokens=400, stage="answer_generation")
        avg_score = sum(s for _, s in retrieved_chunks) / len(retrieved_chunks)

        return {
            "answer": answer,
            "sources": sources,
            "confidence": float(avg_score),
            "chunks_used": len(retrieved_chunks),
        }
    except Exception as e:
        print(f"Answer generation error: {e}")
        return {
            "answer": f"Error generating answer: {e}",
            "sources": sources,
            "confidence": 0.0,
            "chunks_used": len(retrieved_chunks),
        }

## The Full Pipeline in One Class

In [ ]:
# ============================================
# STEP 9: Document Store -- the full pipeline in one class
# ============================================
# EnhancedDocumentStore wires all the previous steps together:
#
#   Document Input -> OCR -> Chunking -> Embeddings -> Vector Storage
#                  -> Retrieval -> LLM Response
#
# process_pdf() runs the left half (input through storage) once per file.
# query() runs the right half (retrieval and answering) once per question.
#
# The Gradio UI and the evaluation both drive this same class, so the numbers
# reported in the evaluation describe the system users actually get.
#
# You'll see: no output here.
# ============================================

class EnhancedDocumentStore:
    """
    Manages the complete document processing and retrieval pipeline.
    """

    def __init__(self):
        self.pages_info: List[PageInfo] = []
        self.logical_docs: List[LogicalDocument] = []
        self.chunks_metadata: List[ChunkMetadata] = []
        self.retriever = IntelligentRetriever()
        self.is_ready = False
        self.processing_stats: Dict = {}
        self.filename: Optional[str] = None

    def process_pdf(self, pdf_file, filename: str = "document.pdf",
                    verbose: bool = True) -> Tuple[bool, Dict]:
        """
        Complete PDF processing pipeline: input -> OCR -> chunking ->
        embeddings -> vector storage.
        """
        self.filename = filename
        self.is_ready = False
        start_time = time.time()

        try:
            # Document input + OCR
            self.pages_info, self.logical_docs = extract_and_analyze_pdf(
                pdf_file, filename=filename, verbose=verbose)

            # Chunking
            self.chunks_metadata = process_all_documents(self.logical_docs, verbose=verbose)

            # Embeddings + vector storage
            self.retriever.build_indices(self.chunks_metadata, verbose=verbose)

            process_time = time.time() - start_time
            n_ocr_pages = sum(1 for p in self.pages_info if p.text_source == "ocr")

            self.processing_stats = {
                "filename": filename,
                "total_pages": len(self.pages_info),
                "pages_from_text_layer": len(self.pages_info) - n_ocr_pages,
                "pages_from_ocr": n_ocr_pages,
                "documents_found": len(self.logical_docs),
                "total_chunks": len(self.chunks_metadata),
                "document_types": sorted({d.doc_type for d in self.logical_docs}),
                "processing_time_sec": round(process_time, 2),
                "processing_time": f"{process_time:.1f}s",
            }

            self.is_ready = True
            return True, self.processing_stats

        except Exception as e:
            return False, {"error": str(e)}

    def query(self, question: str, filter_type: Optional[str] = None,
              auto_route: bool = True, k: int = 4, verbose: bool = True) -> Dict:
        """
        Retrieval + LLM response for a single question.
        """
        if not self.is_ready:
            return {
                "answer": "Please upload and process a PDF first.",
                "sources": [],
                "confidence": 0.0,
                "chunks_used": 0,
                "filter_used": "none",
                "routing_predicted_type": None,
                "routing_confidence": None,
                "search_scope": None,
                "retrieval_sec": 0.0,
                "generation_sec": 0.0,
            }

        # Measure retrieval and generation separately for the timing table.
        t_retr = time.time()
        retrieved = self.retriever.retrieve(
            question, k=k, filter_doc_type=filter_type,
            auto_route=auto_route, verbose=verbose)
        retrieval_sec = time.time() - t_retr

        t_gen = time.time()
        result = generate_answer_with_sources(question, retrieved)
        generation_sec = time.time() - t_gen

        routing = self.retriever.last_routing
        result.update({
            "filter_used": filter_type or ("auto" if auto_route else "none"),
            # The routing model's own predicted type and confidence, which is
            # a different thing from the answer's retrieval-based confidence.
            "routing_predicted_type": routing["predicted_type"] if routing else None,
            "routing_confidence": routing["confidence"] if routing else None,
            "search_scope": self.retriever.last_search_scope,
            "retrieval_sec": round(retrieval_sec, 3),
            "generation_sec": round(generation_sec, 3),
        })
        return result

    def chunks_by_doc_type(self) -> Dict[str, int]:
        """How many chunks exist per document type. Used as the denominator
        for Recall@K in the evaluation section."""
        counts = defaultdict(int)
        for c in self.chunks_metadata:
            counts[c.doc_type] += 1
        return dict(counts)

    def get_document_structure(self) -> List[Dict]:
        """
        Get the document structure for UI display. Page numbers are
        1-indexed here and in citations, so the two always agree.
        """
        return [{
            "id": doc.doc_id,
            "type": doc.doc_type,
            "pages": f"{doc.page_start + 1}-{doc.page_end + 1}",
            "chunks": len(doc.chunks) if doc.chunks else 0,
            "source": doc.text_source,
            "preview": doc.text[:200] + ("..." if len(doc.text) > 200 else ""),
        } for doc in self.logical_docs]

## System Architecture

Generated from the live configuration values, so it always reflects what the code
actually does.

In [ ]:
# ============================================
# STEP 9b: System Architecture Summary
# ============================================
# Prints the pipeline flow and a component specification table, generated
# from the actual configuration values above rather than typed by hand.
#
# This is the source for the System Architecture slide. Because it reads the
# real variables, it cannot drift out of date: change the chunk size or swap
# the model, re-run this cell, and the table updates itself.
#
# You'll see: the pipeline flow diagram and a 7-row component table.
# ============================================

PIPELINE_FLOW = (
    "Document Input  ->  OCR  ->  Chunking  ->  Embeddings  ->  "
    "Vector Storage  ->  Retrieval  ->  LLM Response"
)


def architecture_spec() -> pd.DataFrame:
    """Component specification table, read from live configuration."""
    embed_dim = embed_model.get_sentence_embedding_dimension()
    n_params = sum(p.numel() for p in llm_model.parameters()) / 1e9

    rows = [
        {
            "Component": "OCR Engine",
            "Technology Choice": "Tesseract 5 via pytesseract, PyMuPDF for rasterisation",
            "Configuration Details":
                f"Triggered per page when the text layer has <{OCR_MIN_CHARS} characters; "
                f"pages rendered at {OCR_DPI} DPI; English; digital text layer preferred "
                f"when available",
        },
        {
            "Component": "Text Chunking",
            "Technology Choice": "Custom sliding window (LlamaIndex SentenceSplitter available as an alternative)",
            "Configuration Details":
                f"{CHUNK_SIZE_WORDS}-word chunks, {CHUNK_OVERLAP_WORDS}-word overlap; "
                f"chunks never span two sub-documents; each chunk carries doc type, "
                f"exact page range, source file and digital/OCR flag",
        },
        {
            "Component": "Embeddings",
            "Technology Choice": f"sentence-transformers/{EMBED_MODEL_NAME}",
            "Configuration Details":
                f"{embed_dim}-dimensional vectors, ~22M parameters, L2-normalised so "
                f"inner product equals cosine similarity; runs on {device}",
        },
        {
            "Component": "Vector Database",
            "Technology Choice": "FAISS (in-memory, IndexFlatIP)",
            "Configuration Details":
                "Cosine similarity; exhaustive flat search (exact, no approximation); "
                "one global index plus one index per document type to support filtering",
        },
        {
            "Component": "Retriever",
            "Technology Choice": "Dense vector search with LLM query routing",
            "Configuration Details":
                f"top-k = 4 by default (1-10 in the UI); routing to a single document type "
                f"when the LLM's confidence exceeds {ROUTING_CONFIDENCE_THRESHOLD}, "
                f"otherwise global search; no reranking stage",
        },
        {
            "Component": "LLM",
            "Technology Choice": f"{LLM_MODEL_NAME} (local, Hugging Face Transformers)",
            "Configuration Details":
                f"~{n_params:.1f}B parameters, "
                f"{'float16' if device == 'cuda' else 'float32'} on {device}; greedy decoding "
                f"(temperature 0) for reproducibility; max 400 new tokens for answers, "
                f"20-60 for classification and routing",
        },
        {
            "Component": "Prompt Strategy",
            "Technology Choice": "Zero-shot instruction prompts with labelled context injection",
            "Configuration Details":
                "Retrieved passages are prefixed with [From DocType, Pages X-Y] labels the "
                "model is told to copy verbatim, which makes citations machine-checkable; "
                "an explicit refusal instruction covers questions the context cannot answer",
        },
    ]
    return pd.DataFrame(rows)


print("PIPELINE FLOW\n")
print("   " + PIPELINE_FLOW + "\n")
print("Implemented by EnhancedDocumentStore: process_pdf() covers input through vector")
print("storage, query() covers retrieval through response.\n")
print("COMPONENT SPECIFICATIONS\n")

architecture_df = architecture_spec()
display(architecture_df.style.set_properties(**{"text-align": "left", "white-space": "pre-wrap"}))

## The Gradio Interface

In [ ]:
# ============================================
# STEP 10: Gradio Interface
# ============================================
# The web UI: upload a PDF on the left, document breakdown and retrieval
# settings in the middle, chat on the right. Answers come back with their
# sources, similarity scores and timings attached.
#
# The point of the UI is that a quality engineer can ask "what is the
# expiration date on the Certificate of Quality?" without knowing anything
# about vectors or embeddings.
#
# You'll see: no output here. Step 11 launches it.
# ============================================

# Global store instance shared by the UI callbacks.
doc_store = EnhancedDocumentStore()


def process_pdf_handler(pdf_file):
    """Handle PDF upload and processing."""
    if pdf_file is None:
        return "Please upload a PDF file.", "", gr.update(choices=["All"], value="All")

    if isinstance(pdf_file, str):
        filename = os.path.basename(pdf_file)
    else:
        filename = os.path.basename(getattr(pdf_file, "name", "uploaded.pdf"))

    success, stats = doc_store.process_pdf(pdf_file, filename=filename)

    if not success:
        return f"**Error:** {stats.get('error', 'Unknown error')}", "", gr.update(choices=["All"], value="All")

    status_msg = f"""
**Successfully Processed**
- File: {stats['filename']}
- Pages: {stats['total_pages']} ({stats['pages_from_ocr']} via OCR)
- Documents Found: {stats['documents_found']}
- Chunks Created: {stats['total_chunks']}
- Types: {', '.join(stats['document_types'])}
- Time: {stats['processing_time']}
"""

    structure_display = "\n".join(
        f"- **{doc['type']}** (Pages {doc['pages']}, {doc['source']}): {doc['chunks']} chunks"
        for doc in doc_store.get_document_structure()
    )

    doc_types = ["All"] + stats["document_types"]
    return status_msg, structure_display, gr.update(choices=doc_types, value="All")


def chat_handler(message, history, doc_filter, auto_route, num_chunks):
    """Handle chat interactions."""
    history = history or []

    if not message or not message.strip():
        return history

    if not doc_store.is_ready:
        response = "Please upload and process a PDF document first."
        return history + [{"role": "user", "content": message},
                          {"role": "assistant", "content": response}]

    filter_type = None if doc_filter == "All" else doc_filter
    result = doc_store.query(
        message,
        filter_type=filter_type,
        auto_route=auto_route and filter_type is None,
        k=int(num_chunks),
    )

    response = f"{result['answer']}\n\n"

    if result["sources"]:
        response += "**Sources:**\n"
        for src in result["sources"]:
            file_note = f" — {src['source_file']}" if src.get("source_file") else ""
            ocr_note = " *(OCR)*" if src.get("text_source") == "ocr" else ""
            response += (f"- {src['doc_type']} (Pages {src['pages']}){file_note}{ocr_note}"
                         f" — Similarity: {src['relevance']}\n")

    routing_note = ""
    if result.get("routing_predicted_type"):
        routing_note = (f" | Routed to: {result['routing_predicted_type']} "
                        f"({result['routing_confidence']:.0%})")

    response += (
        f"\n*Mean similarity: {result['confidence']:.1%} | "
        f"Chunks used: {result.get('chunks_used', len(result['sources']))} | "
        f"Scope: {result.get('search_scope', result['filter_used'])}{routing_note} | "
        f"Retrieval {result.get('retrieval_sec', 0):.2f}s, "
        f"generation {result.get('generation_sec', 0):.2f}s*"
    )

    return history + [{"role": "user", "content": message},
                      {"role": "assistant", "content": response}]


def create_interface():
    """Create the Gradio interface for pharmaceutical document Q&A."""

    # Note: in Gradio 6 the theme belongs on launch(), not on Blocks.
    with gr.Blocks(title="Pharmaceutical Document Q&A System") as demo:
        gr.Markdown("""
        # Pharmaceutical Document Q&A System
        ### Intelligent Multi-Document Analysis with a Local RAG Pipeline
        Upload a pharmaceutical blob PDF, click **Process Document** to identify the
        sub-documents inside it and build a searchable index, then ask questions in
        natural language. Answers cite the document type and page they came from.
        """)

        with gr.Row():
            # Left - PDF upload
            with gr.Column(scale=2):
                pdf_input = gr.File(
                    label="Upload Pharmaceutical PDF",
                    file_types=[".pdf"],
                    type="filepath",
                )

                with gr.Row():
                    process_btn = gr.Button("Process Document", variant="primary", size="lg", scale=2)
                    clear_all_btn = gr.Button("Clear All", variant="secondary", size="lg", scale=1)

            # Middle - Document info and settings
            with gr.Column(scale=1):
                gr.Markdown("### Document Info")
                status_output = gr.Markdown(value="Waiting for PDF upload...")
                structure_output = gr.Markdown(value="")

                gr.Markdown("### Retrieval Settings")

                doc_filter = gr.Dropdown(
                    choices=["All"], value="All",
                    label="Document Type Filter",
                    info="Restrict search to one document type",
                )
                auto_route = gr.Checkbox(
                    value=True,
                    label="Auto-Route Queries",
                    info="Let the model pick the most relevant document type",
                )
                num_chunks = gr.Slider(
                    minimum=1, maximum=10, value=4, step=1,
                    label="Chunks to Retrieve (k)",
                )

            # Right - Chat interface
            with gr.Column(scale=2):
                gr.Markdown("### Ask Questions")
                # Gradio 6 removed the `type` argument: the {"role", "content"}
                # message format is now the only one, which is what the handlers
                # below return.
                chatbot = gr.Chatbot(
                    label="Conversation",
                    height=500,
                    elem_id="chatbot",
                    show_label=False,
                )

                with gr.Row():
                    msg_input = gr.Textbox(
                        label="Ask a question",
                        placeholder="e.g. What is the lot number? What sterilization method was used?",
                        scale=4, show_label=False,
                    )
                    send_btn = gr.Button("Send", scale=1, variant="primary")

                with gr.Row():
                    clear_chat_btn = gr.Button("Clear Chat", size="sm", scale=1)
                    example_btn1 = gr.Button("Summarise this document", size="sm", scale=1)
                    example_btn2 = gr.Button("Find lot numbers", size="sm", scale=1)

        with gr.Row():
            status_bar = gr.Markdown(
                value="**Status:** Ready | **Documents:** 0 | **Chunks:** 0",
                elem_id="status_bar",
            )

        # ---------------- event handlers ----------------

        def update_status_bar():
            if doc_store.is_ready:
                stats = doc_store.processing_stats
                return (f"**Status:** Ready | "
                        f"**Documents:** {stats.get('documents_found', 0)} | "
                        f"**Chunks:** {stats.get('total_chunks', 0)}")
            return "**Status:** Ready | **Documents:** 0 | **Chunks:** 0"

        def clear_all():
            global doc_store
            doc_store = EnhancedDocumentStore()
            return (None, "Waiting for PDF upload...", "",
                    gr.update(choices=["All"], value="All"), [], "", update_status_bar())

        def process_pdf_with_status(pdf_file):
            status, structure, filter_update = process_pdf_handler(pdf_file)
            return status, structure, filter_update, update_status_bar()

        def chat_with_status(message, history, doc_filter_v, auto_route_v, num_chunks_v):
            new_history = chat_handler(message, history, doc_filter_v, auto_route_v, num_chunks_v)
            return new_history, update_status_bar()

        def ask_example(question, history, doc_filter_v, auto_route_v, num_chunks_v):
            return chat_handler(question, history, doc_filter_v, auto_route_v, num_chunks_v)

        # ---------------- wiring ----------------

        process_btn.click(
            fn=process_pdf_with_status,
            inputs=[pdf_input],
            outputs=[status_output, structure_output, doc_filter, status_bar],
        )

        clear_all_btn.click(
            fn=clear_all,
            outputs=[pdf_input, status_output, structure_output, doc_filter,
                     chatbot, msg_input, status_bar],
        )

        chat_inputs = [msg_input, chatbot, doc_filter, auto_route, num_chunks]

        msg_input.submit(
            fn=chat_with_status, inputs=chat_inputs, outputs=[chatbot, status_bar]
        ).then(lambda: "", outputs=[msg_input])

        send_btn.click(
            fn=chat_with_status, inputs=chat_inputs, outputs=[chatbot, status_bar]
        ).then(lambda: "", outputs=[msg_input])

        clear_chat_btn.click(lambda: [], outputs=[chatbot])

        example_btn1.click(
            fn=lambda h, f, a, n: ask_example(
                "Summarise the main points of these documents.", h, f, a, n),
            inputs=[chatbot, doc_filter, auto_route, num_chunks],
            outputs=[chatbot],
        ).then(fn=update_status_bar, outputs=[status_bar])

        example_btn2.click(
            fn=lambda h, f, a, n: ask_example(
                "What lot numbers or batch numbers are mentioned in these documents?", h, f, a, n),
            inputs=[chatbot, doc_filter, auto_route, num_chunks],
            outputs=[chatbot],
        ).then(fn=update_status_bar, outputs=[status_bar])

    return demo


print("Gradio interface defined. Run Step 11 to launch it.")

## Launch

In [ ]:
# ============================================
# STEP 11: Launch the Application
# ============================================
# Starts the Gradio app and prints a public URL you can open in a browser
# or screenshot for the presentation.
#
# debug=False keeps this cell from blocking the notebook, so you can run
# everything above and still come back and re-run cells. Set it to True if
# you want live server logs printed here while you click around.
#
# You'll see: a https://....gradio.live link and an inline app frame.
# Upload a PDF, click Process Document, then ask questions.
# ============================================

demo = create_interface()
demo.launch(share=True, debug=False, theme=gr.themes.Soft())